In [1]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"  # Disable MPS memory limits

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"MPS built: {torch.backends.mps.is_built()}")

# Set device for Apple Silicon
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Using MPS device: {device}")
    print(f"MPS memory limit disabled: {os.environ.get('PYTORCH_MPS_HIGH_WATERMARK_RATIO')}")
else:
    device = torch.device("cpu")
    print(f"Using CPU device: {device}")


PyTorch version: 2.8.0
MPS available: True
MPS built: True
Using MPS device: mps
MPS memory limit disabled: 0.0


In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
token = os.getenv("HUGGINGFACE_TOKEN")

if token:
    login(token=token)
    print("✅ Successfully logged into Hugging Face")
else:
    print("⚠️  No Hugging Face token found. Please set HUGGINGFACE_TOKEN in your .env file")


✅ Successfully logged into Hugging Face


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Use a smaller model for macOS memory constraints
model_id = "meta-llama/Llama-2-7b-hf"  # Much smaller model (~345M parameters)
# Alternative: "meta-llama/Llama-2-7b-hf" if you have enough memory

print(f"Using model: {model_id}")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Tokenizer model max length: {tokenizer.model_max_length}")

# Add padding token if it doesn't exist
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


Using model: meta-llama/Llama-2-7b-hf


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-2-7b-hf.
403 Client Error. (Request ID: Root=1-68c91304-0273eb744341bdd443c9ab21;144a6613-4b4b-43da-a165-eef6c468e77c)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-2-7b-hf/resolve/main/config.json.
Access to model meta-llama/Llama-2-7b-hf is restricted and you are not in the authorized list. Visit https://huggingface.co/meta-llama/Llama-2-7b-hf to ask for access.

In [ ]:
from datasets import load_dataset
from transformers import TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
import torch
from torch.utils.data import Dataset
import json

# Load the ARC dataset
raw = load_dataset("allenai/ai2_arc", "ARC-Easy")["train"]
print(f"Loaded {len(raw)} examples from ARC-Easy dataset")

# Use a smaller subset for memory-constrained training on macOS
raw = raw.select(range(min(200, len(raw))))  # Reduced from 500 to 200
print(f"Using {len(raw)} examples for training")


In [ ]:
def format_arc_example(data):
    """Format ARC example into prompt and full_text for training"""
    question = data["question"]
    choices = data["choices"]["text"]
    answer = data.get("answerKey")

    # Format choices
    formatted_choices = "\n".join(f"{chr(ord('A')+i)}. {txt}" for i, txt in enumerate(choices))

    # Messages for prompt only (system + user)
    messages_prompt = [
        {"role": "system", "content": "You are a careful multiple-choice solver."},
        {"role": "user", "content": f"Question: {question}\nChoices:\n{formatted_choices}\nAnswer with a single letter (A-D) only."}
    ]

    # Messages with assistant answer
    messages_full = messages_prompt + [
        {"role": "assistant", "content": answer}
    ]

    prompt_text = tokenizer.apply_chat_template(messages_prompt, tokenize=False, add_generation_prompt=False)
    full_text   = tokenizer.apply_chat_template(messages_full, tokenize=False, add_generation_prompt=False)

    return {"prompt": prompt_text, "full_text": full_text}

formatted_data = [format_arc_example(raw[i]) for i in range(len(raw))]
print(f"Formatted {len(formatted_data)} examples")

print("\nSample prompt:\n", formatted_data[0]["prompt"][:300], "...")
print("\nSample full text:\n", formatted_data[0]["full_text"][:300], "...")



In [ ]:
class ARCDataset(Dataset):
    """
    Expects each item to be one of:
      { "prompt": <string without assistant answer>,
        "full_text": <prompt + assistant answer> }
    or legacy:
      { "text": <prompt + assistant answer> }  # will try to infer prompt via delimiter
    """
    def __init__(self, data, tokenizer, max_length=1024, assistant_delim="Assistant:"):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.assistant_delim = assistant_delim

    def __len__(self):
        return len(self.data)

    def _split_legacy_text(self, text):
        """Best-effort split if only 'text' is provided."""
        if self.assistant_delim in text:
            # use the last occurrence to guard against 'Assistant:' in question text
            cut = text.rfind(self.assistant_delim)
            prompt = text[:cut]
            full_text = text  # already prompt + answer
        else:
            # fallback: treat entire thing as 'full_text' and no prompt (won't mask)
            prompt = ""
            full_text = text
        return prompt, full_text

    def __getitem__(self, idx):
        item = self.data[idx]

        if "prompt" in item and "full_text" in item:
            prompt_text = item["prompt"]
            full_text   = item["full_text"]
        elif "text" in item:
            prompt_text, full_text = self._split_legacy_text(item["text"])
        else:
            raise ValueError("Item must contain either ('prompt' & 'full_text') or 'text'.")

        # 1) Encode PROMPT ONLY (no padding) to get true prompt length in tokens
        enc_prompt = self.tokenizer(
            prompt_text,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
            padding=False,
        )

        # 2) Encode FULL TEXT (prompt + assistant answer) with padding for the model
        enc_full = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
            padding="max_length",
        )

        input_ids = enc_full["input_ids"].squeeze(0)
        attn_mask = enc_full["attention_mask"].squeeze(0)

        labels = input_ids.clone()
        prompt_len = enc_prompt["input_ids"].size(1) if enc_prompt["input_ids"].ndim == 2 else int(enc_prompt["input_ids"].shape[-1])

        # Mask out everything before the assistant's answer
        labels[:prompt_len] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attn_mask,
            "labels": labels,
        }


# Create dataset
train_dataset = ARCDataset(formatted_data, tokenizer, max_length=1024)
print(f"Created training dataset with {len(train_dataset)} examples")


In [ ]:
# Load model for fine-tuning with memory optimization
print("Loading model with memory optimizations...")

# Clear any existing memory
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

# Load model with memory-efficient settings
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,  # Use float32 for better MPS compatibility
    device_map="auto" if not torch.backends.mps.is_available() else None,
    low_cpu_mem_usage=True,     # Reduce memory usage during loading
    attn_implementation="eager"  # Use eager attention for better MPS compatibility
)

# Move model to device with memory management
if torch.backends.mps.is_available():
    try:
        model = model.to(device)
        print(f"✅ Model loaded on MPS device: {next(model.parameters()).device}")
    except RuntimeError as e:
        if "out of memory" in str(e):
            print("⚠️  MPS out of memory, falling back to CPU")
            device = torch.device("cpu")
            model = model.to(device)
            print(f"✅ Model loaded on CPU device: {next(model.parameters()).device}")
        else:
            raise e
else:
    model = model.to(device)
    print(f"✅ Model loaded on device: {next(model.parameters()).device}")

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

from peft import LoraConfig, get_peft_model, TaskType

# --- Auto-discover viable target modules ---
targets = set()
for name, _ in model.named_modules():
    if name.endswith("c_attn"):
        targets.add("c_attn")
    if name.endswith("c_proj"):
        targets.add("c_proj")

print("LoRA targets detected:", targets)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=sorted(targets),  # <- discovered automatically
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# Set MPS memory management
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

training_args = TrainingArguments(
    output_dir="./output",
    num_train_epochs=1,
    per_device_train_batch_size=1,              # Small batch size for macOS
    gradient_accumulation_steps=4,              # Reduced from 8 to save memory
    warmup_steps=10,
    learning_rate=5e-5,
    logging_steps=1,
    save_steps=50,
    eval_strategy="no",                         # Changed from evaluation_strategy
    save_strategy="steps",
    load_best_model_at_end=False,
    report_to=None,                             # Disable wandb/tensorboard
    use_mps_device=torch.backends.mps.is_available(),
    dataloader_pin_memory=False,                # Disable pin memory for macOS
    remove_unused_columns=False,
    fp16=False,                                 # fp16 not supported on MPS
    bf16=False,                                 # bf16 not supported on MPS
    dataloader_num_workers=0,                   # Reduce memory usage
    max_grad_norm=1.0,                          # Gradient clipping to help with memory
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
)

# Memory cleanup before training
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

print("🚀 Starting training...")
print(f"Training on device: {device}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation steps: {training_args.gradient_accumulation_steps}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Dataset size: {len(train_dataset)} examples")

trainer.train()
print("✅ Training completed!")


In [ ]:
# Test the fine-tuned model
def test_model(model, tokenizer, question, choices):
    formatted_choices = "\n".join(f"{chr(65+i)}. {c}" for i, c in enumerate(choices))
    messages = [
        {"role":"system","content":"You are a careful multiple-choice solver."},
        {"role":"user","content":f"Question: {question}\nChoices:\n{formatted_choices}\nAnswer with a single letter (A-D) only."}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt")
    if torch.backends.mps.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=1,         # <- only one token
            do_sample=False,          # <- greedy
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Extract just the new token
    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    letter = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    # Optional: sanitize to A-D
    letter = (letter[:1].upper() if letter else "")
    if letter not in ["A","B","C","D"]:
        letter = "A"  # or empty; up to you
    return letter


# Test with a sample question
if len(formatted_data) > 0:
    sample = raw[0]  # Get original sample
    print("Sample question:")
    print(f"Question: {sample['question']}")
    print(f"Choices: {sample['choices']['text']}")
    print(f"Correct answer: {sample['answerKey']}")
    
    # Test the model
    response = test_model(model, tokenizer, sample['question'], sample['choices']['text'])
    print(f"\nModel response:\n{response}")
    
    print("\n Note: This is a simplified test. For production use, implement proper evaluation metrics.")
